# M02A — Classification Workflow & Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M02-Foundations/M02A-Classification-Workflow-and-Features.ipynb)

**Pattern Classification practical · approximately 2 hours · M02 · CLO1, CLO2**

This is the canonical student-facing practical for session `M02A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1B-00-MLPipeline.ipynb`, `F1B-03-MLPolyFeaGeneration.ipynb`, `F1B-04-MLSampleGenerators.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

A classifier is only one component of a trustworthy workflow. You will generate controlled data, preserve a final test set, place feature construction inside a pipeline, and compare the result with a baseline.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Construct a leakage-safe classification workflow using a scikit-learn pipeline.
- Generate and interpret informative, redundant, noisy, and interaction features.
- Compare a learned classifier with a defensible baseline using held-out evidence.
- Diagnose common shape, leakage, and reproducibility problems.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- an executed leakage-safe classification pipeline
- a baseline and cross-validation comparison
- a justified feature-complexity choice and short reflection

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Python functions and arrays
- Basic probability and classification terminology
- Familiarity with train/test splits

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

A supervised classification workflow separates **representation**, **estimation**, and **evaluation**. Any transform that learns from data must be fitted inside the training process. A `Pipeline` makes that ordering explicit and prevents information from the validation or test set leaking into preprocessing.

Polynomial features can express interactions such as $x_1x_2$, but they increase dimensionality and may amplify overfitting. Cross-validation estimates how a workflow—not just its last estimator—generalises to unseen samples.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. Generate data with known structure

`make_classification` lets us control informative, redundant, and noisy features. Predict which columns a linear model will find useful before running the cell.

In [ ]:
X, y = make_classification(
    n_samples=800, n_features=6, n_informative=3, n_redundant=1,
    n_clusters_per_class=2, class_sep=1.0, flip_y=0.04,
    random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("train:", X_train.shape, "test:", X_test.shape)
print("positive-class rates:", y_train.mean().round(3), y_test.mean().round(3))

### 2. Establish a baseline

A useful model should beat a strategy that only predicts the most frequent training class.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline test accuracy: {baseline_accuracy:.3f}")

### 3. Put feature generation inside the workflow

The transform is fitted separately inside each cross-validation fold. Compare the training and validation means; a large gap is an overfitting warning.

In [ ]:
workflow = Pipeline([
    ("features", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_validate(workflow, X_train, y_train, cv=cv, return_train_score=True)
print(f"CV train accuracy: {scores['train_score'].mean():.3f}")
print(f"CV validation accuracy: {scores['test_score'].mean():.3f} ± {scores['test_score'].std():.3f}")

### 4. Fit once on all training data and evaluate once on the test set

The held-out set is used only after the workflow choices have been made.

In [ ]:
workflow.fit(X_train, y_train)
test_predictions = workflow.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Workflow test accuracy: {test_accuracy:.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, cmap="Blues")
plt.title("M02A held-out confusion matrix")
plt.show()

## Student Challenge

Compare polynomial degrees 1, 2, and 3 using the same five folds. Choose one degree using validation evidence only, evaluate it once on the held-out test set, and explain the trade-off between model flexibility and generalisation.

In [ ]:
# Starter scaffold: extend rather than delete this record of results.
challenge_results = []
for degree in [1, 2, 3]:
    # TODO: build a leakage-safe pipeline and evaluate it with `cv` on X_train/y_train.
    # Append: {"degree": degree, "mean_cv_accuracy": ..., "std_cv_accuracy": ...}
    pass

challenge_results

### Challenge success criteria

- All preprocessing is inside each candidate pipeline.
- The degree is selected without consulting `y_test`.
- The final claim cites both validation variation and held-out performance.
- The notebook is reproducible from a fresh kernel.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
# Exercise missing-information and failure behaviour.
try:
    workflow.predict(np.empty((1, X_train.shape[1] - 1)))
except ValueError as error:
    print("Expected wrong-feature-count failure:", str(error).split("\n")[0])
else:
    raise AssertionError("A wrong feature count should fail clearly.")

assert np.isfinite(X_train).all(), "Handle missing values before this workflow."
print("Edge check: the guided dataset contains no missing or infinite values.")

In [ ]:
assert X_train.shape[1] == 6
assert len(X_train) + len(X_test) == len(X)
assert set(workflow.named_steps) == {"features", "scale", "classifier"}
assert 0.0 <= baseline_accuracy <= 1.0
assert 0.0 <= test_accuracy <= 1.0
assert test_accuracy > baseline_accuracy, "Investigate the random seed, data, or pipeline."
print("M02A guided-practice checks passed.")

## Extension Task

Repeat the challenge with class imbalance (`weights=[0.85, 0.15]`). Compare accuracy with balanced accuracy and explain why the preferred metric can change.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Source notebooks listed at the top of this lab.
- scikit-learn User Guide: [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- scikit-learn User Guide: [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)